# MNQ objective strategy validation

This notebook runs the MNQ objective strategy on deterministic in-memory fixtures. The fixture data is synthetic and research-only: it is useful for checking contracts, ordering, costs, risk guards, and reporting, but it is not market data and does not establish profitability or live execution quality.

The default path requires no downloads, credentials, broker connection, API access, or external files.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import polars as pl
from IPython.display import display

repo_root = next(
    (
        root
        for root in (Path.cwd(), *Path.cwd().parents)
        if (root / "research" / "mnq_strategy").is_dir()
    ),
    None,
)
if repo_root is None:
    raise RuntimeError("Run this notebook from the repository root or a descendant directory.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from research.mnq_strategy.backtest import run_backtest
from research.mnq_strategy.config import StrategyConfig
from research.mnq_strategy.evaluation import WalkForwardWindow, walk_forward_evaluate
from research.mnq_strategy.fixtures import make_confirmed_signal_fixture, make_multi_month_fixture

## Reproducible setup

The primary backtest uses a compact normalized fixture with one closed confirmation and a next-bar entry. A separate chronological multi-month fixture supplies non-empty train/test rows for the walk-forward report. Both are deterministic Polars frames from `research.mnq_strategy.fixtures`.

In [ ]:
config = StrategyConfig()
config.validate_fixed_contract()

bars = make_confirmed_signal_fixture()
walk_forward_bars = make_multi_month_fixture()

print("Data source: deterministic in-memory MNQ fixture (synthetic/research-only)")
print(f"Instrument: {config.instrument}")
print(f"Bars for primary backtest: {bars.height}")
print(f"Bars for walk-forward validation: {walk_forward_bars.height}")
print(f"Config hash: {config.config_hash}")

## Backtest signals and trades

Signals are confirmed on closed bars and entries are observed on the next chronological bar. The result table keeps rejected signals alongside closed trades so the first 20 rows remain auditable.

In [ ]:
backtest_results = run_backtest(bars, config)

print("First 20 input signal rows:")
display(bars.filter(pl.col("signal")).head(20))

print("First 20 backtest signal/trade rows:")
display(backtest_results.head(20))

## Equity curve and drawdown

The curve uses realized `net_pnl` from closed trades, including the configured commission and slippage model. If matplotlib is unavailable, the same curve and drawdown values are shown as a table.

In [ ]:
closed_trades = backtest_results.filter(pl.col("status") == "closed").sort("exit_time")
if closed_trades.height:
    baseline = pl.DataFrame(
        {
            "exit_time": pl.Series("exit_time", [None], dtype=closed_trades.schema["exit_time"]),
            "net_pnl": pl.Series("net_pnl", [0.0], dtype=pl.Float64),
        }
    )
    equity_curve = (
        pl.concat([baseline, closed_trades.select("exit_time", "net_pnl")], how="vertical")
        .with_columns(pl.col("net_pnl").cum_sum().alias("equity"))
        .with_columns((pl.col("equity").cum_max() - pl.col("equity")).alias("drawdown"))
    )
    assert equity_curve["equity"][0] == 0.0
    assert equity_curve["drawdown"].max() == 36.0
else:
    equity_curve = pl.DataFrame({"exit_time": [], "net_pnl": [], "equity": [], "drawdown": []})

print("Equity and drawdown table:")
display(equity_curve)

try:
    import matplotlib.pyplot as plt
except (ImportError, ModuleNotFoundError, RuntimeError):
    plt = None

if plt is None:
    print("matplotlib is unavailable; the table above is the plotting fallback.")
else:
    equity_observation = list(range(equity_curve.height))
    figure, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
    axes[0].plot(equity_observation, equity_curve["equity"].to_list(), marker="o")
    axes[0].set_title("Synthetic fixture equity curve")
    axes[0].set_ylabel("USD")
    axes[0].grid(alpha=0.25)
    axes[1].plot(
        equity_observation, equity_curve["drawdown"].to_list(), marker="o", color="tab:red"
    )
    axes[1].set_title("Drawdown")
    axes[1].set_xlabel("Equity observation")
    axes[1].set_ylabel("USD")
    axes[1].grid(alpha=0.25)
    figure

## Fixed-contract walk-forward validation

The thresholds are fixed by `StrategyConfig`; the test window is not used to select parameters. The printed report includes the configuration hash, window metadata, selection policy, the train/test `chronology_check`, and explicit `signal_provenance_check='not_available'` metadata. The chronology result is not signal-level no-lookahead evidence.

In [ ]:
task7_windows = {
    "July 2024": WalkForwardWindow("2024-01-01", "2024-05-31", "2024-07-01", "2024-07-31"),
    "September 2024": WalkForwardWindow("2024-01-01", "2024-07-31", "2024-09-01", "2024-09-30"),
    "November 2024": WalkForwardWindow("2024-01-01", "2024-09-30", "2024-11-01", "2024-11-30"),
    "Holdout": WalkForwardWindow("2024-01-01", "2024-10-31", "2024-11-01", "2024-11-30"),
}
task7_reports = {
    name: walk_forward_evaluate(walk_forward_bars, [window], config)
    for name, window in task7_windows.items()
}
task7_metrics = {
    name: {
        key: report[key]
        for key in (
            "trade_count",
            "net_pnl",
            "expectancy",
            "win_rate",
            "profit_factor",
            "max_drawdown",
            "daily_loss_breaches",
            "consecutive_loss_breaches",
            "average_r",
            "cost_share",
            "per_setup",
            "chronology_check",
            "signal_provenance_check",
            "config_hash",
        )
    }
    for name, report in task7_reports.items()
}
print("Task 7 metrics (report regeneration output):")
print(json.dumps(task7_metrics, indent=2, sort_keys=True, default=str))

assert all(report["config_hash"] == config.config_hash for report in task7_reports.values())
assert all(report["chronology_check"] is True for report in task7_reports.values())
assert all(
    report["signal_provenance_check"] == "not_available" for report in task7_reports.values()
)

## Daily guard execution fixture

This cell regenerates the executable risk-guard evidence. The fixture contains no hand-written PnL column: the two losses are produced by the configured stop and costs, and the third signal is rejected after the daily loss limit is reached.

In [ ]:
from research.mnq_strategy.fixtures import make_daily_guard_fixture

daily_guard_results = run_backtest(make_daily_guard_fixture(), config)
daily_guard_closed = daily_guard_results.filter(pl.col("status") == "closed")
daily_guard_rejections = daily_guard_results.filter(pl.col("status") == "rejected")

assert daily_guard_closed["net_pnl"].to_list() == [-225.0, -225.0]
assert daily_guard_rejections["rejection_reason"].to_list() == ["daily_loss_limit"]
print("Daily guard closed losses:", daily_guard_closed["net_pnl"].to_list())
print("Daily guard rejection:", daily_guard_rejections["rejection_reason"].to_list())

This is a local research smoke workflow only. Replace the fixture with a locally normalized MNQ file only after applying the same timestamp, closed-bar, signal, cost, and evaluation contracts; do not interpret this synthetic run as evidence of profitability.